# 03 - Exploratory Data Analysis (EDA)
In this notebook, we perform a deep-dive exploratory data analysis on the newly updated datasets to uncover over 10 actionable dimensions of ride cancellations, pricing, time of day, turnaround times, and location dynamics.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_style('whitegrid')


## 1. Load Datasets & Initial Check
Loading both the generalized clean data and the specialized cancelled data.


In [ ]:
cleaned_df = pd.read_csv('../data/processed/cleaned_dataset.csv')
cancelled_df = pd.read_csv('../data/processed/cancelled_dataset.csv')

# Create a binary target column for Cancellation
cleaned_df['Is_Cancelled'] = cleaned_df['Booking_Status'].apply(lambda x: 0 if x == 'Success' else 1)

print("Cleaned Data Shape:", cleaned_df.shape)
print("Cancelled Data Shape:", cancelled_df.shape)
display(cleaned_df.head())


## 2. Global Cancellation Rates vs Raw Counts
Evaluating core categories (Peak Hours, Vehicle Type, Payment Method, and Hourly) by their cancellation rate (%), not just raw count volume.


In [ ]:
def plot_cancellation_rate(df, column, title):
    rate_df = df.groupby(column)['Is_Cancelled'].mean().reset_index()
    rate_df['Is_Cancelled'] *= 100
    rate_df = rate_df.sort_values(by='Is_Cancelled', ascending=False)
    
    plt.figure(figsize=(10, 5))
    sns.barplot(data=rate_df, x=column, y='Is_Cancelled', palette='Reds_r')
    plt.title(title)
    plt.ylabel('Cancellation Rate (%)')
    plt.xticks(rotation=45)
    plt.show()

plot_cancellation_rate(cleaned_df, 'Is_Peak_Hour', 'Cancellation Rate by Peak Hour')
plot_cancellation_rate(cleaned_df, 'Vehicle_Type', 'Cancellation Rate by Vehicle Type')
plot_cancellation_rate(cleaned_df, 'Payment_Method', 'Cancellation Rate by Payment Method')
plot_cancellation_rate(cleaned_df, 'Hour', 'Cancellation Rate by Hour of Day')


## 3. Correlation Heatmap
Mapping linear relationships between continuous/numerical variables (Distance, Ratings, Turnaround Times, and Cancellations).


In [ ]:
numerical_cols = ['Ride_Distance', 'Booking_Value', 'Driver_Ratings', 'Customer_Rating', 'Hour', 'V_TAT', 'C_TAT', 'Is_Cancelled']
corr_matrix = cleaned_df[[c for c in numerical_cols if c in cleaned_df.columns]].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Correlation Heatmap of Numerical Variables')
plt.show()


## 4. Day-of-Week & Weekend Analysis
How do specific days of the week, and weekends overall, dictate reliability?


In [ ]:
if 'Day_of_Week' in cleaned_df.columns:
    days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    cleaned_df['Day_of_Week'] = pd.Categorical(cleaned_df['Day_of_Week'], categories=days_order, ordered=True)
    plot_cancellation_rate(cleaned_df, 'Day_of_Week', 'Cancellation Rate by Day of Week')

if 'Is_Weekend' in cleaned_df.columns:
    plot_cancellation_rate(cleaned_df, 'Is_Weekend', 'Cancellation Rate: Weekday vs Weekend')


## 5. Time of Day Patterns
Looking beyond just 'peak hours' to broader 'Time of Day' segments (Morning, Evening, Night).


In [ ]:
if 'Time_of_Day' in cleaned_df.columns:
    plot_cancellation_rate(cleaned_df, 'Time_of_Day', 'Cancellation Rate by Time of Day')


## 6. Vehicle Turnaround Time (V_TAT) Impact
Does longer vehicle ETA (Turnaround Time) correlate with higher Customer Cancellations?


In [ ]:
if 'V_TAT' in cleaned_df.columns:
    plt.figure(figsize=(10, 6))
    status_filter = cleaned_df[cleaned_df['Booking_Status'].isin(['Success', 'Canceled By Driver', 'Canceled By Customer'])]
    sns.boxplot(data=status_filter, x='Booking_Status', y='V_TAT', palette='Pastel1')
    plt.title('Vehicle Turnaround Time (V_TAT) by Booking Status')
    plt.ylabel('V_TAT (Wait Time)')
    plt.show()


## 7. Driver vs Customer Cancellations by Distance
Does a longer requested ride distance trigger more Driver Cancellations compared to Customer Cancellations?


In [ ]:
plt.figure(figsize=(10, 6))
status_filter = cleaned_df[cleaned_df['Booking_Status'].isin(['Success', 'Canceled By Driver', 'Canceled By Customer'])]
sns.boxplot(data=status_filter, x='Booking_Status', y='Ride_Distance', palette='Set2')
plt.title('Ride Distance Distribution by Booking Status')
plt.ylabel('Ride Distance (km)')
plt.show()


## 8. Booking Value per KM (Pricing) Impact
Is there a pricing elasticity where highly-priced trips (per KM) have distinct cancellation behaviours?


In [ ]:
if 'Booking_Value_per_KM' in cleaned_df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=status_filter, x='Booking_Status', y='Booking_Value_per_KM', palette='Set3')
    plt.title('Booking Value per KM by Booking Status')
    plt.ylabel('Booking Value per KM')
    plt.ylim(0, cleaned_df['Booking_Value_per_KM'].quantile(0.95)) # Cap outliers for visibility
    plt.show()


## 9. Customer and Driver Ratings
Do chronically low-rated customers get cancelled on more often by drivers? Do low-rated drivers get cancelled on by customers?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(ax=axes[0], data=status_filter, x='Booking_Status', y='Customer_Rating', palette='cool')
axes[0].set_title('Customer Rating by Booking Status')

sns.boxplot(ax=axes[1], data=status_filter, x='Booking_Status', y='Driver_Ratings', palette='spring')
axes[1].set_title('Driver Rating by Booking Status')

plt.tight_layout()
plt.show()


## 10. Cancellation Reason Breakdown
Analyzing the raw qualitative reasons for cancellations from the secondary cancelled dataset.


In [ ]:
if 'Canceled_Rides_by_Driver' in cancelled_df.columns:
    driver_reasons = cancelled_df['Canceled_Rides_by_Driver'].replace('N/A', np.nan).dropna().value_counts()
    plt.figure(figsize=(10, 5))
    sns.barplot(y=driver_reasons.index, x=driver_reasons.values, palette='Blues_r')
    plt.title('Top Reasons for Driver Cancellations')
    plt.xlabel('Count')
    plt.show()

if 'Canceled_Rides_by_Customer' in cancelled_df.columns:
    customer_reasons = cancelled_df['Canceled_Rides_by_Customer'].replace('N/A', np.nan).dropna().value_counts()
    plt.figure(figsize=(10, 5))
    sns.barplot(y=customer_reasons.index, x=customer_reasons.values, palette='Oranges_r')
    plt.title('Top Reasons for Customer Cancellations')
    plt.xlabel('Count')
    plt.show()


## 11. Location Hotspots
Identifying the top 10 Pickup Locations by volume and observing their respective cancellation rates.


In [ ]:
top_10_locations = cleaned_df['Pickup_Location'].value_counts().head(10).index
loc_filter = cleaned_df[cleaned_df['Pickup_Location'].isin(top_10_locations)]

loc_rates = loc_filter.groupby('Pickup_Location')['Is_Cancelled'].mean().reset_index()
loc_rates['Is_Cancelled'] *= 100
loc_rates = loc_rates.sort_values(by='Is_Cancelled', ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=loc_rates, x='Pickup_Location', y='Is_Cancelled', palette='magma')
plt.title('Cancellation Rate across Top 10 Busiest Pickup Locations')
plt.ylabel('Cancellation Rate (%)')
plt.xticks(rotation=45)
plt.show()


## 12. Cancellation Trends over the Month (Day of Month)
Plotting how cancellations evolve day-by-day throughout the recorded month.


In [ ]:
if 'Day' in cleaned_df.columns:
    # Calculate raw counts and cancellation rate per day
    day_stats = cleaned_df.groupby('Day').agg(
        Total_Bookings=('Is_Cancelled', 'count'),
        Cancellation_Rate=('Is_Cancelled', lambda x: x.mean() * 100)
    ).reset_index()

    fig, ax1 = plt.subplots(figsize=(14, 6))
    
    # Plot Total Bookings as a bar chart on ax1
    sns.barplot(data=day_stats, x='Day', y='Total_Bookings', color='lightblue', ax=ax1, alpha=0.6)
    ax1.set_ylabel('Total Bookings (Count)', color='blue')
    ax1.tick_params(axis='y', labelcolor='blue')
    
    # Plot Cancellation Rate as a line graph on ax2
    ax2 = ax1.twinx()
    sns.lineplot(data=day_stats, x='Day', y='Cancellation_Rate', color='red', marker='o', linewidth=2, ax=ax2)
    ax2.set_ylabel('Cancellation Rate (%)', color='red')
    ax2.tick_params(axis='y', labelcolor='red')
    
    plt.title('Total Bookings and Cancellation Rate by Day of Month')
    plt.show()


## Executive Summary of EDA Findings
- **Temporal Factors:** Cancellation rates vary starkly by hour, day of week, day of month, and peak classification.
- **Distance & Value:** Driver cancellations frequently skew towards specific ride distances, and higher wait times (V_TAT) correlate strongly with booking failure.
- **Qualitative Context:** Driver cancellations and Customer cancellations have wildly different operational distributions and root causes, proving that any ride allocation model must treat them as distinct phenomena.
